# Reporte Meteorologico

- Temperatura actual via **WeatherAPI.com** (estaciones fisicas reales)
- Historial de 60 dias via **Open-Meteo**
- Prediccion de proximas 3 horas (intervalos de 30 min) via **Random Forest (ML)**

In [ ]:
!pip install requests pandas scikit-learn --quiet

In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

WEATHERAPI_KEY = 'TU_API_KEY_AQUI'  # Regístrate gratis en weatherapi.com
LAT               = 19.5046
LON               = -99.1463
CIUDAD            = 'ESCOM, Gustavo A. Madero, Ciudad de Mexico'
MINUTOS_INTERVALO = 30
INTERVALOS        = 6   # 6 x 30 min = proximas 3 horas
DIAS_HISTORICOS   = 60
BASE_URL_METEO    = 'https://api.open-meteo.com/v1'

print('Librerias')

In [ ]:
def obtener_temperatura_actual():
    url = 'http://api.weatherapi.com/v1/current.json'
    params = {'key': WEATHERAPI_KEY, 'q': f'{LAT},{LON}', 'lang': 'es'}
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    data = r.json()
    c = data['current']
    return {
        'temperature_2m':       c['temp_c'],
        'apparent_temperature': c['feelslike_c'],
        'relative_humidity_2m': c['humidity'],
        'wind_speed_10m':       c['wind_kph'],
        'surface_pressure':     c['pressure_mb'],
        'dew_point_2m':         c['dewpoint_c'],
        'condicion':            c['condition']['text'],
        'time':                 data['location']['localtime']
    }

def obtener_historico():
    params = {
        'latitude': LAT, 'longitude': LON,
        'hourly': ('temperature_2m,relative_humidity_2m,wind_speed_10m,'
                   'apparent_temperature,surface_pressure,shortwave_radiation,dew_point_2m'),
        'timezone': 'America/Mexico_City',
        'past_days': DIAS_HISTORICOS,
        'forecast_days': 1
    }
    r = requests.get(f'{BASE_URL_METEO}/forecast', params=params, timeout=20)
    r.raise_for_status()
    df = pd.DataFrame(r.json()['hourly'])
    df['time'] = pd.to_datetime(df['time'])
    return df.dropna()

def crear_features(df):
    df = df.copy()
    df['hora']       = df['time'].dt.hour
    df['dia_semana'] = df['time'].dt.dayofweek
    df['mes']        = df['time'].dt.month
    df['hora_sin']   = np.sin(2 * np.pi * df['hora'] / 24)
    df['hora_cos']   = np.cos(2 * np.pi * df['hora'] / 24)
    df['mes_sin']    = np.sin(2 * np.pi * df['mes']  / 12)
    df['mes_cos']    = np.cos(2 * np.pi * df['mes']  / 12)
    for lag in [1, 2, 3, 6, 12, 24]:
        df[f'temp_lag_{lag}h'] = df['temperature_2m'].shift(lag)
    df['media_3h']   = df['temperature_2m'].rolling(3).mean()
    df['media_6h']   = df['temperature_2m'].rolling(6).mean()
    df['media_12h']  = df['temperature_2m'].rolling(12).mean()
    df['delta_1h']   = df['temperature_2m'].diff(1)
    df['delta_3h']   = df['temperature_2m'].diff(3)
    df['rango_dia']  = df.groupby(df['time'].dt.date)['temperature_2m'].transform(lambda x: x.max() - x.min())
    df['hr_x_rocio'] = df['relative_humidity_2m'] * df['dew_point_2m']
    return df.dropna()

COLUMNAS = [
    'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos', 'dia_semana',
    'relative_humidity_2m', 'wind_speed_10m', 'surface_pressure',
    'shortwave_radiation', 'dew_point_2m',
    'temp_lag_1h', 'temp_lag_2h', 'temp_lag_3h',
    'temp_lag_6h', 'temp_lag_12h', 'temp_lag_24h',
    'media_3h', 'media_6h', 'media_12h',
    'delta_1h', 'delta_3h', 'rango_dia', 'hr_x_rocio'
]

def entrenar_modelo(df):
    df_feat = crear_features(df)
    X, y = df_feat[COLUMNAS], df_feat['temperature_2m']
    corte = int(len(X) * 0.8)
    modelo = RandomForestRegressor(
        n_estimators=400, max_depth=14, min_samples_leaf=2,
        max_features=0.75, random_state=42, n_jobs=-1
    )
    modelo.fit(X.iloc[:corte], y.iloc[:corte])
    mae = mean_absolute_error(y.iloc[corte:], modelo.predict(X.iloc[corte:]))
    return modelo, mae, df_feat

def predecir_intervalos(modelo, df_feat):
    ultima = df_feat.iloc[-1].copy()
    temps  = list(df_feat['temperature_2m'].tail(24))
    predicciones = []
    ahora = datetime.now()
    for i in range(1, INTERVALOS + 1):
        t    = ahora + timedelta(minutes=MINUTOS_INTERVALO * i)
        hora = t.hour + t.minute / 60.0
        fila = {
            'hora_sin':             np.sin(2 * np.pi * hora / 24),
            'hora_cos':             np.cos(2 * np.pi * hora / 24),
            'mes_sin':              np.sin(2 * np.pi * t.month / 12),
            'mes_cos':              np.cos(2 * np.pi * t.month / 12),
            'dia_semana':           t.weekday(),
            'relative_humidity_2m': ultima['relative_humidity_2m'],
            'wind_speed_10m':       ultima['wind_speed_10m'],
            'surface_pressure':     ultima['surface_pressure'],
            'shortwave_radiation':  (max(0, ultima['shortwave_radiation'] *
                                    np.sin(np.pi * hora / 14)) if 6 <= hora <= 20 else 0),
            'dew_point_2m':         ultima['dew_point_2m'],
            'temp_lag_1h':          temps[-1],
            'temp_lag_2h':          temps[-2],
            'temp_lag_3h':          temps[-3],
            'temp_lag_6h':          temps[-6],
            'temp_lag_12h':         temps[-12] if len(temps) >= 12 else temps[0],
            'temp_lag_24h':         temps[-24] if len(temps) >= 24 else temps[0],
            'media_3h':             np.mean(temps[-3:]),
            'media_6h':             np.mean(temps[-6:]),
            'media_12h':            np.mean(temps[-12:]) if len(temps) >= 12 else np.mean(temps),
            'delta_1h':             temps[-1] - temps[-2],
            'delta_3h':             temps[-1] - temps[-3] if len(temps) >= 3 else 0,
            'rango_dia':            ultima['rango_dia'],
            'hr_x_rocio':           ultima['relative_humidity_2m'] * ultima['dew_point_2m'],
        }
        pred = modelo.predict(pd.DataFrame([fila])[COLUMNAS])[0]
        predicciones.append((t, round(pred, 1)))
        temps.append(pred)
    return predicciones

print('Funciones')

In [ ]:
def separador():
    print('-' * 52)

print('Consultando datos...', end='\r')

actual               = obtener_temperatura_actual()
temp_actual          = actual['temperature_2m']
df                   = obtener_historico()
modelo, mae, df_feat = entrenar_modelo(df)
predicciones         = predecir_intervalos(modelo, df_feat)
temps_pred           = [t for _, t in predicciones]
tendencia            = 'Al alza' if temps_pred[-1] > temp_actual else 'A la baja'
confianza            = 'Alta' if mae < 0.8 else 'Media' if mae < 1.5 else 'Baja'

separador()
print(f'  REPORTE METEOROLOGICO')
print(f'  {CIUDAD}')
separador()
print(f"  Fecha y hora de observacion   {actual['time']}")
print(f"  Condicion atmosferica         {actual['condicion']}")
separador()
print(f'  Temperatura actual            {temp_actual} C')
print(f"  Sensacion termica             {actual['apparent_temperature']} C")
print(f"  Humedad relativa              {actual['relative_humidity_2m']} %")
print(f"  Punto de rocio                {actual['dew_point_2m']} C")
print(f"  Velocidad del viento          {actual['wind_speed_10m']} km/h")
print(f"  Presion atmosferica           {actual['surface_pressure']} hPa")
separador()
print(f'  PRONOSTICO — PROXIMOS {INTERVALOS * MINUTOS_INTERVALO} MINUTOS (cada {MINUTOS_INTERVALO} min)')
separador()
print(f"  {'Hora':<12} {'Temperatura estimada':>20}")
for tiempo, temp in predicciones:
    print(f"  {tiempo.strftime('%H:%M hrs'):<12} {str(temp) + ' C':>20}")
separador()
print(f'  Minima estimada               {min(temps_pred)} C')
print(f'  Maxima estimada               {max(temps_pred)} C')
print(f'  Tendencia                     {tendencia}')
separador()
print(f'  Temperatura actual (fuente)   WeatherAPI.com')
print(f'  Historial y entrenamiento     Open-Meteo (open-meteo.com)')
print(f'  Modelo                        Random Forest  /  {len(COLUMNAS)} variables')
print(f'  Error promedio (MAE)          {mae:.2f} C  —  Confianza {confianza}')
print(f'  Coordenadas                   {LAT} N,  {LON} W')
separador()